# 03 — PU Learning + 5-Fold LightGBM

Final modeling pipeline using the **27 engineered features**:
PU learning with spies → iterative reliable-negative mining → confidence-weighted
LightGBM → 5-fold OOF threshold tuning → fold-ensemble prediction.

Legacy commented code and the later 10-feature ablation experiment were removed.

In [ ]:
from pathlib import Path
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df_ref = pd.read_parquet(ARTIFACT_DIR / "df_feature.parquet")
df_features = pd.read_parquet(ARTIFACT_DIR / "df_feature_final.parquet")

train_accts = df_ref.loc[df_ref["split"] == "train", ["acct", "label"]].drop_duplicates("acct")
df_feat_map = df_features.set_index("acct")
feature_cols = df_feat_map.columns.tolist()

if len(feature_cols) != 27:
    raise ValueError(f"Expected 27 final features, found {len(feature_cols)}.")

print(f"Train accounts: {len(train_accts):,}")
print(f"Positive: {(train_accts['label'] == 1).sum():,}")
print(f"Unlabeled: {(train_accts['label'] == 0).sum():,}")

In [ ]:
N_SPLITS = 5
RANDOM_STATE = 42
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

LGBM_PARAMS = dict(
    objective="binary",
    metric="average_precision",
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=63,
    min_child_samples=80,
    min_sum_hessian_in_leaf=1e-3,
    min_split_gain=0.2,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    lambda_l1=0.0,
    lambda_l2=5.0,
    max_bin=255,
    class_weight=None,
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

In [ ]:
def pu_train_and_score(X, core_p_idx, u_spy_idx, random_state):
    pu_model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=120,
        learning_rate=0.08,
        max_depth=4,
        num_leaves=31,
        class_weight="balanced",
        random_state=random_state,
        verbose=-1,
    )
    X_p = X.iloc[core_p_idx]
    X_u = X.iloc[u_spy_idx]
    pu_model.fit(
        pd.concat([X_p, X_u]),
        np.concatenate([np.ones(len(X_p), dtype=int), np.zeros(len(X_u), dtype=int)]),
    )
    return pu_model.predict_proba(X_u)[:, 1], pu_model


def pick_reliable_negatives(scores, u_spy_idx, spy_idx, target_k, spy_quantile, min_sep_gap):
    if len(spy_idx):
        is_spy = np.isin(u_spy_idx, spy_idx)
        spy_scores = scores[is_spy]
        u_scores = scores[~is_spy]
        u_idx = u_spy_idx[~is_spy]
    else:
        spy_scores = np.array([])
        u_scores = scores
        u_idx = u_spy_idx

    use_threshold = len(spy_scores) and len(u_scores)
    if use_threshold:
        threshold = np.quantile(spy_scores, spy_quantile)
        gap = float(np.mean(spy_scores) - np.mean(u_scores))
        use_threshold = gap >= min_sep_gap

    if use_threshold:
        candidate_mask = u_scores < threshold
        candidate_idx = u_idx[candidate_mask]
        candidate_scores = u_scores[candidate_mask]
        if len(candidate_idx) >= target_k:
            return candidate_idx[np.argsort(candidate_scores)[:target_k]]
        need = target_k - len(candidate_idx)
        ranked = u_idx[np.argsort(u_scores)]
        ranked = ranked[~np.isin(ranked, candidate_idx)]
        return np.concatenate([candidate_idx, ranked[:need]])

    return u_idx[np.argsort(u_scores)[:target_k]]

In [ ]:
def iterative_rn_selection(
    X, p_idx, u_idx, rng_seed,
    spy_frac=0.05, rn_rounds=3, rn_schedule=(0.02, 0.01, 0.005),
    min_rn_per_round=300, max_total_rn=15000,
    spy_quantile=0.10, min_sep_gap=0.02,
):
    rng = np.random.default_rng(rng_seed)
    spy_n = max(1, int(len(p_idx) * spy_frac))
    spy_n = min(spy_n, len(p_idx) - 1) if len(p_idx) > 1 else 0
    spy_idx = rng.choice(p_idx, size=spy_n, replace=False) if spy_n else np.array([], dtype=int)
    core_p_idx = np.setdiff1d(p_idx, spy_idx)

    remaining_u = u_idx.copy()
    rn_rounds_list = []
    last_pu_model = None

    for r in range(rn_rounds):
        if not len(remaining_u):
            break

        ratio = rn_schedule[r] if r < len(rn_schedule) else rn_schedule[-1]
        target_k = max(min_rn_per_round, int(len(remaining_u) * ratio))
        u_spy_idx = np.concatenate([remaining_u, spy_idx]) if len(spy_idx) else remaining_u

        scores, last_pu_model = pu_train_and_score(
            X, core_p_idx, u_spy_idx, random_state=rng_seed + 100 + r
        )
        rn_idx = pick_reliable_negatives(
            scores, u_spy_idx, spy_idx, target_k, spy_quantile, min_sep_gap
        )

        rn_rounds_list.append(rn_idx)
        remaining_u = remaining_u[~np.isin(remaining_u, rn_idx)]
        print(f"  round {r + 1}: RN={len(rn_idx):,}, remaining U={len(remaining_u):,}")

        if sum(len(x) for x in rn_rounds_list) >= max_total_rn:
            break

    final_rn_idx = np.concatenate(rn_rounds_list) if rn_rounds_list else np.array([], dtype=int)
    return p_idx, final_rn_idx, last_pu_model


def build_sample_weight(X, p_idx, rn_idx, pu_model, pos_weight=15.0, rn_weight_floor=1.0):
    weights = np.ones(len(p_idx) + len(rn_idx), dtype=float)
    weights[:len(p_idx)] = pos_weight

    if pu_model is not None and len(rn_idx):
        rn_scores = pu_model.predict_proba(X.iloc[rn_idx])[:, 1]
        rn_scores = np.nan_to_num(rn_scores, nan=1.0, posinf=1.0, neginf=0.0)
        weights[len(p_idx):] = np.clip(2.0 - rn_scores, rn_weight_floor, 2.0)

    return weights

In [ ]:
def train_fold_model(X, p_idx, rn_idx, X_val, y_val, sample_weight):
    X_train = pd.concat([X.iloc[p_idx], X.iloc[rn_idx]])
    y_train = np.concatenate([
        np.ones(len(p_idx), dtype=int),
        np.zeros(len(rn_idx), dtype=int),
    ])

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_val, y_val)],
        eval_metric="average_precision",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    metrics = {
        "f1": f1_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "best_iteration": model.best_iteration_,
    }
    return model, proba, metrics


def mean_feature_importance(models):
    rows = []
    for model in models:
        booster = model.booster_
        gain = booster.feature_importance(importance_type="gain")
        rows.append(pd.DataFrame({
            "feature": booster.feature_name(),
            "gain": gain,
            "gain_pct": gain / (gain.sum() + 1e-12),
        }))
    return (
        pd.concat(rows)
        .groupby("feature")[["gain", "gain_pct"]]
        .mean()
        .sort_values("gain", ascending=False)
        .reset_index()
    )


def tune_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    best = int(np.argmax(f1))
    threshold = thresholds[best] if best < len(thresholds) else 0.5
    return float(threshold), float(f1[best])

In [ ]:
accounts = train_accts["acct"].values
labels = train_accts["label"].values
groups = accounts

cv_rows, fold_models = [], []
all_val_probs, all_val_labels = [], []

for fold, (tr_idx, va_idx) in enumerate(splitter.split(accounts, labels, groups), 1):
    print(f"\nFOLD {fold}")
    train_accounts, val_accounts = accounts[tr_idx], accounts[va_idx]
    y_train, y_val = labels[tr_idx], labels[va_idx]

    X_train_full = df_feat_map.reindex(train_accounts)[feature_cols]
    X_val = df_feat_map.reindex(val_accounts)[feature_cols]
    p_idx = np.where(y_train == 1)[0]
    u_idx = np.where(y_train == 0)[0]

    final_p_idx, final_rn_idx, pu_model = iterative_rn_selection(
        X_train_full, p_idx, u_idx, rng_seed=RANDOM_STATE + fold
    )
    sample_weight = build_sample_weight(
        X_train_full, final_p_idx, final_rn_idx, pu_model
    )
    model, proba, metrics = train_fold_model(
        X_train_full, final_p_idx, final_rn_idx, X_val, y_val, sample_weight
    )

    fold_models.append(model)
    all_val_probs.append(proba)
    all_val_labels.append(y_val)
    cv_rows.append({
        "fold": fold,
        **metrics,
        "P": len(final_p_idx),
        "RN": len(final_rn_idx),
    })

cv_df = pd.DataFrame(cv_rows)
feature_importance = mean_feature_importance(fold_models)
y_true_all = np.concatenate(all_val_labels)
y_proba_all = np.concatenate(all_val_probs)
best_threshold, best_f1 = tune_threshold(y_true_all, y_proba_all)

print(cv_df)
print(f"OOF best threshold: {best_threshold:.6f}")
print(f"OOF best F1: {best_f1:.4f}")

cv_df.to_csv(ARTIFACT_DIR / "cv_results.csv", index=False)
feature_importance.to_csv(ARTIFACT_DIR / "feature_importance.csv", index=False)

In [ ]:
top_features = (
    feature_importance.set_index("feature")["gain_pct"]
    .sort_values()
    .tail(27)
)
plt.figure(figsize=(10, 8))
plt.barh(top_features.index, top_features.values)
plt.xlabel("Average gain importance")
plt.title("Feature Importance Across 5 Folds")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Final prediction uses the same 27 features and averages probabilities across folds.
predict_accts = df_ref.loc[df_ref["split"] == "test", ["acct"]].drop_duplicates("acct")
acct_list = predict_accts["acct"].values
X_pred = (
    df_feat_map.reindex(acct_list)[feature_cols]
    .fillna(0)
    .replace([np.inf, -np.inf], 0)
)

proba_mean = np.mean(
    [model.predict_proba(X_pred)[:, 1] for model in fold_models],
    axis=0,
)
labels_pred = (proba_mean >= best_threshold).astype(int)

submission = pd.DataFrame({"acct": acct_list, "label": labels_pred})
submission.to_csv(ARTIFACT_DIR / "submission.csv", index=False, encoding="utf-8-sig")
print(f"Prediction accounts: {len(submission):,}")
print(f"Predicted positives: {submission['label'].sum():,}")